# YOLOv8-style 2D+t Training

Notebook nay train ban YOLOv8-style custom cho bai toan 2D+t:

- input: `Tensor[B, 6, 448, 448]`
- output: `Tensor[B, 7, 7, 15]`
- motion: `mx, my, mw, mh`


In [ ]:
from pathlib import Path

REPO_DIR = Path('/content/YOLO2D-t')
if REPO_DIR.exists():
    print('Repo da ton tai:', REPO_DIR)
else:
    !git clone https://github.com/thangSy221105/YOLO2D-t.git /content/YOLO2D-t
%cd /content/YOLO2D-t


In [ ]:
!pip install -q -r requirements.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!find /content/drive/MyDrive -maxdepth 6 -type d | grep "processed\|YOLO 2D+t\|dataset_YOLO" || true


In [ ]:
from pathlib import Path

# Sua path nay neu folder processed cua ban khac.
PROCESSED_DIR = Path('/content/drive/MyDrive/dataset_YOLO/YOLO 2D+t/YOLO 2D+t/data/processed')

print('PROCESSED_DIR =', PROCESSED_DIR)
print('processed exists =', PROCESSED_DIR.exists())
print('pairs dir exists =', (PROCESSED_DIR / 'pairs').exists())
print('annotations dir exists =', (PROCESSED_DIR / 'annotations').exists())


## Optional: copy data sang `/content`

Neu doc truc tiep tu Drive cham, chay cell nay roi doi `PROCESSED_DIR = Path('/content/data/processed')` o cell tiep theo.

In [ ]:
# Optional speed-up. Bo comment neu can copy data tu Drive sang /content.
# import shutil
# DRIVE_ROOT = PROCESSED_DIR.parent.parent
# SRC_PROCESSED = DRIVE_ROOT / 'data' / 'processed'
# SRC_MOT17 = DRIVE_ROOT / 'data' / 'MOT17'
# DST_PROCESSED = Path('/content/data/processed')
# DST_MOT17 = Path('/content/data/MOT17')
# if DST_PROCESSED.exists():
#     shutil.rmtree(DST_PROCESSED)
# if DST_MOT17.exists():
#     shutil.rmtree(DST_MOT17)
# shutil.copytree(SRC_PROCESSED, DST_PROCESSED)
# shutil.copytree(SRC_MOT17, DST_MOT17)
# PROCESSED_DIR = DST_PROCESSED
# print('Using local PROCESSED_DIR =', PROCESSED_DIR)


In [ ]:
from pathlib import Path
import textwrap

config_text = f"""
seed: 42

data:
  image_size: 448
  grid_size: 7
  boxes_per_cell: 2
  num_classes: 1
  batch_size: 8
  num_workers: 2
  delta: 40
  processed_dir: {PROCESSED_DIR.as_posix()}
  train_split: train
  val_split: val
  use_horizontal_flip: false
  return_meta: false
  pin_memory: true

model:
  in_channels: 6
  width: 0.5
  depth: 0.33
  dropout: 0.1
  activate_output: true

loss:
  lambda_coord: 5.0
  lambda_noobj: 0.5
  lambda_class: 1.0
  lambda_motion: 1.0

train:
  epochs: 10
  lr: 1.0e-4
  weight_decay: 1.0e-4
  device: cuda
  mixed_precision: true
  grad_clip_norm: 10.0
  log_interval: 10
  output_dir: outputs/yolov8_2dt
  save_every_epoch: true
"""

config_path = Path('configs/yolov8_2dt_colab.yaml')
config_path.write_text(textwrap.dedent(config_text).strip() + '\n', encoding='utf-8')
print(config_path.read_text(encoding='utf-8'))


In [ ]:
import sys
sys.path.insert(0, 'src')

from yolo2dt.config import load_config
from yolo2dt.data_adapter import build_dataloaders
from yolo2dt.yolov8_model import YoloV8Style2DT

cfg = load_config('configs/yolov8_2dt_colab.yaml')
train_loader, val_loader = build_dataloaders(cfg)
batch = next(iter(train_loader))
images = batch['image'] if isinstance(batch, dict) else batch[0]
targets = batch['target'] if isinstance(batch, dict) else batch[1]
motion_masks = batch['motion_mask'] if isinstance(batch, dict) else batch[2]

print('image shape =', tuple(images.shape))
print('target shape =', tuple(targets.shape))
print('motion_mask shape =', tuple(motion_masks.shape))

model = YoloV8Style2DT(
    in_channels=cfg['model']['in_channels'],
    grid_size=cfg['data']['grid_size'],
    boxes_per_cell=cfg['data']['boxes_per_cell'],
    num_classes=cfg['data']['num_classes'],
    width=cfg['model']['width'],
    depth=cfg['model']['depth'],
)
preds = model(images[:1])
print('pred shape =', tuple(preds.shape))


In [ ]:
!python train_yolov8_2dt.py --config configs/yolov8_2dt_colab.yaml


In [ ]:
!python visualize_yolov8_2dt.py --config configs/yolov8_2dt_colab.yaml --conf 0.3 --output outputs/yolov8_2dt/prediction_preview.jpg

from IPython.display import Image, display
display(Image(filename='outputs/yolov8_2dt/prediction_preview.jpg'))
